### openslr-52-asr-sinhala — clean

Applies the cleaning pipeline (audio-hash dedup, VAD-based silence trim) directly to the
**original raw OpenSLR-52 file tree** (`data/raw/openslr_52/asr_sinhala/`, loose `.flac` files
+ `utt_spk_text.tsv`), instead of the `openslrData` parquet conversion.

The raw source is never modified. Each stage writes its result into its own new folder under
`data/processed/`, mirroring the raw tree's layout (`data/<2-char-prefix>/<file_id>.flac` +
`utt_spk_text.tsv`):

- `openslr_52_dedup/` -- deduplicated copy (kept files only, unmodified audio)
- `openslr_52_clean/` -- final copy: deduplicated + VAD-trimmed audio, plus `meta.tsv`
  (per-file VAD stats and trim amounts)

Since every file is read individually from disk (not one giant parquet blob), there's no need
for the batched/streamed `pyarrow` writer trick the parquet-based version needed to avoid OOM --
each stage is just a plain per-file loop.

## 0. Setup

In [7]:
!pip install -q pandas numpy soundfile tqdm torch torchaudio matplotlib noisereduce

## 1. Reconcile the raw corpus

`utt_spk_text.tsv` lists `file_id`, `speaker_id`, `transcript`. Each `file_id`'s audio lives at
`data/<file_id[:2]>/<file_id>.flac` (confirmed against the tree on disk). This just builds the
working file list -- dropping any TSV rows with no matching `.flac` -- everything downstream
reads from `df`.

In [8]:
import os

import pandas as pd
from IPython.display import Audio, display
from tqdm.auto import tqdm

RAW_DIR = "../data/raw/openslr_52/asr_sinhala"
RAW_TSV = os.path.join(RAW_DIR, "utt_spk_text.tsv")
RAW_DATA_DIR = os.path.join(RAW_DIR, "data")


def flac_path(file_id, root):
    return os.path.join(root, file_id[:2], f"{file_id}.flac")


tsv_df = pd.read_csv(RAW_TSV, sep="\t", header=None, names=["file_id", "speaker_id", "transcript"])
tsv_df["path"] = [flac_path(fid, RAW_DATA_DIR) for fid in tsv_df["file_id"]]

exists_mask = [os.path.exists(p) for p in tqdm(tsv_df["path"], desc="checking files on disk")]
missing = (~pd.Series(exists_mask)).sum()
if missing:
    print(f"[warn] {missing} TSV rows have no matching .flac on disk -- dropping them")
df = tsv_df[exists_mask].reset_index(drop=True)

print(f"Reconciled rows: {len(df):,} (of {len(tsv_df):,} in TSV)")
df.head(3)

/home/yohan-jayasinghe/WhisperSL/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
checking files on disk: 100%|██████████| 155970/155970 [00:00<00:00, 443359.96it/s]

Reconciled rows: 155,970 (of 155,970 in TSV)


,file_id,speaker_id,transcript,path
0,0000f47c22,7ab05,මහවැලි ගඟට ගොස් ආපසු එන ගමනේදී,../data/raw/openslr_52/asr_sinhala/data/00/000...
1,000101700f,44e28,උන්වහන්සේ කපාපු,../data/raw/openslr_52/asr_sinhala/data/00/000...
2,000107b539,b1a64,එය එතනින් අවසන් නොවී,../data/raw/openslr_52/asr_sinhala/data/00/000...


## 2. Dedup pass (audio-hash dedup, copied to a new folder)

Reads and decodes each file once (the decode is needed for the content hash anyway, so duration
is derived from the same decode instead of a second `sf.info` call). First occurrence of a given
`audio_hash` is copied into `openslr_52_dedup/`; every later occurrence is dropped and counted.
`meta_df` holds per-file stats (no audio bytes) for the checks below.

In [9]:
import hashlib
import shutil

import soundfile as sf

DEDUP_DIR = "../data/raw/openslr_52/processed/openslr_52_dedup"
DEDUP_DATA_DIR = os.path.join(DEDUP_DIR, "data")
DEDUP_TSV = os.path.join(DEDUP_DIR, "utt_spk_text.tsv")
os.makedirs(DEDUP_DATA_DIR, exist_ok=True)

N_AUDIT_PAIRS = 5  # duplicate-audio pairs to audition later


def audio_hash_and_duration(path):
    samples, sr = sf.read(path, dtype="int16", always_2d=True)
    mono = samples.mean(axis=1).astype("int16")
    h = hashlib.md5(mono.tobytes() + str(sr).encode()).hexdigest()
    return h, len(mono) / sr


seen_hash_id = {}   # audio_hash -> first file_id that had it
meta_records = []   # file_id, duration, text_len, text, audio_hash, kept
dup_examples = []   # up to N_AUDIT_PAIRS of {hash, kept_id, dropped_id}
kept_rows = []       # (file_id, speaker_id, transcript) for kept rows -- written to DEDUP_TSV
n_dropped = 0

for row in tqdm(df.itertuples(index=False), total=len(df), desc="dedup pass"):
    h, dur = audio_hash_and_duration(row.path)
    text_len = len(row.transcript) if pd.notna(row.transcript) else 0

    kept = h not in seen_hash_id
    if kept:
        seen_hash_id[h] = row.file_id
        out_path = flac_path(row.file_id, DEDUP_DATA_DIR)
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        shutil.copy2(row.path, out_path)
        kept_rows.append((row.file_id, row.speaker_id, row.transcript))
    else:
        n_dropped += 1
        if len(dup_examples) < N_AUDIT_PAIRS:
            dup_examples.append({"hash": h, "kept_id": seen_hash_id[h], "dropped_id": row.file_id})

    meta_records.append({"file_id": row.file_id, "duration": dur, "text_len": text_len,
                          "text": row.transcript, "audio_hash": h, "kept": kept})

meta_df = pd.DataFrame(meta_records).set_index("file_id")

pd.DataFrame(kept_rows, columns=["file_id", "speaker_id", "transcript"]).to_csv(
    DEDUP_TSV, sep="\t", header=False, index=False
)

total_rows = len(df)
print(f"\nTotal rows: {total_rows:,}")
print(f"Rows dropped (duplicate audio_hash): {n_dropped:,}")
print(f"Rows kept: {total_rows - n_dropped:,}")
print(f"Total hours (all rows): {meta_df['duration'].sum() / 3600:.2f}")
print(f"Saved deduplicated copy to {DEDUP_DIR}")
meta_df.head(3)

dedup pass: 100%|██████████| 155970/155970 [05:10<00:00, 502.71it/s]



Total rows: 155,970
Rows dropped (duplicate audio_hash): 0
Rows kept: 155,970
Total hours (all rows): 188.99
Saved deduplicated copy to ../data/raw/openslr_52/processed/openslr_52_dedup


,duration,text_len,text,audio_hash,kept
file_id,,,,,
0000f47c22,5.6,30,මහවැලි ගඟට ගොස් ආපසු එන ගමනේදී,213a6922ffdfeed791d6d4a957ce3964,True
000101700f,2.9,15,උන්වහන්සේ කපාපු,6f9a9a74b9344010788f8eef01a0d638,True
000107b539,2.8,20,එය එතනින් අවසන් නොවී,7d3a7b0537efeb30075855fcdf13bea9,True


## 3. Duplicated transcripts (informational)

Purely informational -- the actual dedup above drops by `audio_hash`, not by transcript, since a
shared transcript can legitimately be read by different speakers.

In [10]:
text = meta_df["text"].fillna("").str.strip()

dup_mask = text.duplicated(keep=False) & (text != "")
print(f"Rows with a duplicated transcript: {dup_mask.sum()}")

dup_transcript_groups = meta_df[dup_mask].groupby(text[dup_mask]).groups
print(f"Distinct duplicated transcripts: {len(dup_transcript_groups)}")

for t, idxs in list(dup_transcript_groups.items())[:3]:
    print(f"\n--- {t[:120]}")
    print(meta_df.loc[idxs, ["duration", "text_len"]].to_string())

Rows with a duplicated transcript: 104262
Distinct duplicated transcripts: 42267

--- ' පෝරකයට නංවා ගෙලට තොණ්ඩුව දැමීමෙන්ද පසුව
            duration  text_len
file_id                       
4f5693d1e1       5.6        41
feece35157       5.9        41

--- ''කැළණි පාලම''යි.
            duration  text_len
file_id                       
10b5aa5975       2.8        17
6c4010f475       2.5        17

--- ''තාත්තා කෝ'' යයි
            duration  text_len
file_id                       
3b42ae46a6       6.6        17
403b0300be       3.7        17


## 4. Dedup summary

In [11]:
n_dup_hash_groups = meta_df.loc[~meta_df["kept"], "audio_hash"].nunique()
print(f"Duplicate audio groups: {n_dup_hash_groups}")
print(f"Rows dropped: {n_dropped}")
print(f"{total_rows} -> {total_rows - n_dropped}")
print(f"Hours: {meta_df['duration'].sum() / 3600:.2f} -> "
      f"{meta_df.loc[meta_df['kept'], 'duration'].sum() / 3600:.2f}")

Duplicate audio groups: 0
Rows dropped: 0
155970 -> 155970
Hours: 188.99 -> 188.99


### Audit duplicate-audio pairs

Files are read straight off disk by path -- the kept copy from `openslr_52_dedup/`, the dropped
one from the original raw tree (it was never copied).

In [12]:
for ex in dup_examples:
    k, d = ex["kept_id"], ex["dropped_id"]
    print(f"\n=== hash {ex['hash']}")
    print(f"  kept    [{k}] {meta_df.loc[k, 'duration']:.2f}s  {meta_df.loc[k, 'text'][:80]!r}")
    display(Audio(filename=flac_path(k, DEDUP_DATA_DIR)))
    print(f"  dropped [{d}] {meta_df.loc[d, 'duration']:.2f}s  {meta_df.loc[d, 'text'][:80]!r}")
    display(Audio(filename=flac_path(d, RAW_DATA_DIR)))

## 5. VAD-based silence measurement

Runs Silero VAD over every kept (deduplicated) file. Audio itself doesn't change here, so unlike
the dedup stage this does **not** copy audio anywhere -- it just writes a `vad_meta.tsv` sidecar
next to `openslr_52_dedup/`'s audio, keyed by `file_id`.

In [13]:
import torch
import torchaudio

# torch.hub's fork-validation call to the GitHub API can get rate-limited when unauthenticated
# and crash on a missing header; skip that check.
import torch.hub
torch.hub._validate_not_a_forked_repo = lambda a, b, c: True

silero_model, silero_utils = torch.hub.load(
    "snakers4/silero-vad", "silero_vad", trust_repo=True
)
get_speech_timestamps = silero_utils[0]


def vad_analyse(path, sr_target=16000):
    x, sr = sf.read(path, dtype="float32", always_2d=True)
    x = x.mean(axis=1)
    dur = len(x) / sr

    wav = torch.from_numpy(x)
    if sr != sr_target:
        wav = torchaudio.functional.resample(wav, sr, sr_target)

    ts = get_speech_timestamps(wav, silero_model, sampling_rate=sr_target, return_seconds=True)

    if not ts:
        return dict(duration=dur, vad_silence_frac=1.0, vad_lead_sil_s=dur,
                    vad_trail_sil_s=dur, vad_speech_s=0.0, vad_n_segments=0)

    speech_s = sum(t["end"] - t["start"] for t in ts)

    return dict(
        duration         = dur,
        vad_silence_frac = max(0.0, 1 - speech_s / dur),
        vad_lead_sil_s   = ts[0]["start"],
        vad_trail_sil_s  = max(0.0, dur - ts[-1]["end"]),
        vad_speech_s     = speech_s,
        vad_n_segments   = len(ts),
    )

Using cache found in /home/yohan-jayasinghe/.cache/torch/hub/snakers4_silero-vad_master


In [14]:
N_AUDIT = 3  # audit examples per category (largest lead/trail silence, zero-speech)
VAD_META_PATH = os.path.join(DEDUP_DIR, "vad_meta.tsv")

kept_ids = meta_df.index[meta_df["kept"]].tolist()
print(f"Dedup rows: {len(kept_ids):,}")

vad_records = []
for file_id in tqdm(kept_ids, desc="VAD pass"):
    v = vad_analyse(flac_path(file_id, DEDUP_DATA_DIR))
    vad_records.append({"file_id": file_id, **v})

vad_df = pd.DataFrame(vad_records).set_index("file_id")
vad_df.to_csv(VAD_META_PATH, sep="\t")

n_no_speech = int((vad_df["vad_n_segments"] == 0).sum())
print(f"\nRows with zero VAD speech segments: {n_no_speech}")
print(f"Saved VAD metadata to {VAD_META_PATH}")
vad_df[["duration", "vad_silence_frac", "vad_lead_sil_s", "vad_trail_sil_s",
        "vad_speech_s", "vad_n_segments"]].describe()

Dedup rows: 155,970


VAD pass: 100%|██████████| 155970/155970 [1:06:57<00:00, 38.82it/s]



Rows with zero VAD speech segments: 31
Saved VAD metadata to ../data/raw/openslr_52/processed/openslr_52_dedup/vad_meta.tsv


,duration,vad_silence_frac,vad_lead_sil_s,vad_trail_sil_s,vad_speech_s,vad_n_segments
count,155970.000000,155970.000000,155970.000000,155970.000000,155970.000000,155970.000000
mean,4.362069,0.495429,1.349641,0.798992,2.166930,1.124223
std,1.605085,0.127044,0.914853,0.577465,0.867606,0.400538
min,1.100000,0.092105,0.000000,0.000000,0.000000,0.000000
25%,3.300000,0.402334,0.900000,0.500000,1.600000,1.000000
50%,4.000000,0.487805,1.100000,0.700000,2.000000,1.000000
75%,5.000000,0.583333,1.500000,1.000000,2.600000,1.000000
max,30.700000,1.000000,24.600000,27.300000,12.400000,9.000000


### Audit VAD (before deciding on trim)

Worst offenders picked with a plain `nlargest(...)`; audio is read straight from
`openslr_52_dedup/` by path.

In [15]:
from IPython.display import Audio, display
worst_lead_idx = vad_df["vad_lead_sil_s"].nlargest(N_AUDIT).index.tolist()
worst_trail_idx = vad_df["vad_trail_sil_s"].nlargest(N_AUDIT).index.tolist()
no_speech_idx = vad_df[vad_df["vad_n_segments"] == 0].index[:N_AUDIT].tolist()

audit_idx = sorted(set(worst_lead_idx) | set(worst_trail_idx) | set(no_speech_idx))

for label, idxs in [("largest lead silence", worst_lead_idx), ("largest trail silence", worst_trail_idx)]:
    print(f"\n### {label}")
    for i in idxs:
        row = vad_df.loc[i]
        print(f"\n[{i}] {row['duration']:.2f}s  lead={row['vad_lead_sil_s']:.2f}s  "
              f"trail={row['vad_trail_sil_s']:.2f}s  {meta_df.loc[i, 'text'][:80]!r}")
        display(Audio(filename=flac_path(i, DEDUP_DATA_DIR)))

print(f"\n### zero-speech clips ({n_no_speech} total)")
for i in no_speech_idx:
    row = vad_df.loc[i]
    print(f"\n[{i}] {row['duration']:.2f}s  text: {meta_df.loc[i, 'text'][:80]!r}")
    display(Audio(filename=flac_path(i, DEDUP_DATA_DIR)))


### largest lead silence

[1998607e4d] 26.70s  lead=24.60s  trail=0.70s  'එය තීරණය වෙන්නේ,'



[efbd5eb9ec] 26.80s  lead=21.30s  trail=1.90s  'එවකට විර පරාක්\u200dරම නරේන්ද්\u200dරසිංහ රජතුමා'



[cdc293228d] 25.80s  lead=20.90s  trail=1.40s  'ඉන්දියානු රුපියල් කෝටි ක වියදමක් දරලා'



### largest trail silence

[f0a1800408] 29.70s  lead=1.00s  trail=27.30s  'වැඩ කලොත් අපේ රට'



[2464fc3e0d] 29.30s  lead=0.70s  trail=26.00s  'මීට අවුරුදු පන්දහස් ගානකට කලින්'



[695b9e248e] 27.60s  lead=2.00s  trail=23.40s  'ගමේ මිනිසුන්ට මේ ආරංචිය ගියාම'



### zero-speech clips (31 total)

[0d735b19fb] 3.50s  text: 'cool tamil'



[10087acc1e] 3.10s  text: 'මේ ගැන ලිපියක් පළ කෙරුවා'



[1184f95bdd] 4.40s  text: 'පාලකයින් බලපෑම් කරන්නේනම් එය වැරදියි'


## 6. Trim leading/trailing silence (VAD-based, buffered)

Trims each clip to `[lead_sil - BUFFER_S, duration - trail_sil + BUFFER_S]` rather than flush to
the raw VAD boundary, for the same reason as before: Silero's `min_speech_duration_ms=250` /
`speech_pad_ms=30` defaults mean a brief bit of genuine speech right at an edge can fall under
the floor and not register as a segment, or register with only 30ms of padding.

`MIN_DURATION_S = 3.0` keeps every kept window at least 3s (Whisper fine-tuning's preferred
floor): if the buffered window would come out shorter, it's grown symmetrically around the
speech (clamped to the clip) instead -- the trim never cuts a clip down to less than 3s. Clips
already under 3s are left untouched (kept as-is, not trimmed further), and so are
`vad_n_segments == 0` clips (VAD found no speech anywhere) -- trimming on VAD's own say-so
there would delete the whole clip, so those are flagged for manual audition instead.

This writes the final `.flac` files into `openslr_52_clean/` -- the deliverable copy -- plus its
own `utt_spk_text.tsv` and a `meta.tsv` (VAD stats + trim amounts) alongside it.

In [16]:
BUFFER_S = 0.3  # safety margin kept on each side of the VAD speech boundary, see markdown above
MIN_DURATION_S = 3.0  # Whisper fine-tuning wants 3-30s clips; never trim below this floor


def trim_bounds(duration, lead_sil, trail_sil, n_segments, buffer_s=BUFFER_S, min_duration_s=MIN_DURATION_S):
    if n_segments == 0:
        return 0.0, duration

    speech_start = lead_sil
    speech_end = duration - trail_sil

    start = max(0.0, speech_start - buffer_s)
    end = min(duration, speech_end + buffer_s)

    if start >= end:  # buffer collapsed the window (very short clip) -- bail out, keep original
        return 0.0, duration

    if end - start < min_duration_s:
        if duration <= min_duration_s:
            return 0.0, duration
        deficit = min_duration_s - (end - start)
        start = max(0.0, start - deficit / 2)
        end = min(duration, start + min_duration_s)
        start = max(0.0, end - min_duration_s)  # re-clamp if end hit the clip's end

    return start, end


def trim_and_write(src_path, dst_path, start_s, end_s):
    samples, sr = sf.read(src_path, dtype="int16", always_2d=True)
    i0 = int(round(start_s * sr))
    i1 = int(round(end_s * sr))
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    sf.write(dst_path, samples[i0:i1], sr, format="FLAC", subtype="PCM_16")

In [17]:
import os 
CLEAN_DIR = "../data/raw/openslr_52/processed/openslr_52_clean"
CLEAN_DATA_DIR = os.path.join(CLEAN_DIR, "data")
CLEAN_TSV = os.path.join(CLEAN_DIR, "utt_spk_text.tsv")
CLEAN_META_PATH = os.path.join(CLEAN_DIR, "meta.tsv")
os.makedirs(CLEAN_DATA_DIR, exist_ok=True)

speaker_by_id = {fid: spk for fid, spk, _ in kept_rows}

# Same clips flagged in the VAD audit above -- captures before/after trim for exactly those,
# instead of re-deriving "worst offenders" with a second heap/heuristic.
trim_audit_idx = set(audit_idx)
trim_audit_payload = {}

meta2_records = []
clean_rows = []
n_no_speech_trim = 0

for file_id in tqdm(kept_ids, desc="trim pass"):
    v = vad_df.loc[file_id]
    dur, lead, trail, nseg = v["duration"], v["vad_lead_sil_s"], v["vad_trail_sil_s"], v["vad_n_segments"]

    start, end = trim_bounds(dur, lead, trail, nseg)
    trim_and_write(flac_path(file_id, DEDUP_DATA_DIR), flac_path(file_id, CLEAN_DATA_DIR), start, end)

    if nseg == 0:
        n_no_speech_trim += 1

    if file_id in trim_audit_idx:
        trim_audit_payload[file_id] = {"orig_dur": dur, "trim_dur": end - start, "lead": lead, "trail": trail}

    meta2_records.append({
        "file_id": file_id, "duration": end - start, "duration_pre_trim": dur,
        "vad_silence_frac": v["vad_silence_frac"], "vad_lead_sil_s": lead, "vad_trail_sil_s": trail,
        "vad_speech_s": v["vad_speech_s"], "vad_n_segments": nseg,
        "trimmed_lead_s": start, "trimmed_trail_s": dur - end,
    })
    clean_rows.append((file_id, speaker_by_id[file_id], meta_df.loc[file_id, "text"]))

meta2_df = pd.DataFrame(meta2_records).set_index("file_id")
meta2_df.to_csv(CLEAN_META_PATH, sep="\t")
pd.DataFrame(clean_rows, columns=["file_id", "speaker_id", "transcript"]).to_csv(
    CLEAN_TSV, sep="\t", header=False, index=False
)

print(f"Rows with zero VAD speech segments (left untrimmed, flagged): {n_no_speech_trim}")
print(f"\nTotal audio: {meta2_df['duration_pre_trim'].sum() / 3600:.2f}h -> "
      f"{meta2_df['duration'].sum() / 3600:.2f}h")
print(f"Removed: {(meta2_df['duration_pre_trim'] - meta2_df['duration']).sum() / 3600:.3f}h "
      f"({(meta2_df['trimmed_lead_s'] + meta2_df['trimmed_trail_s']).mean():.2f}s/clip avg)")

n_still_short = (meta2_df["duration"] < MIN_DURATION_S).sum()
print(f"Clips still under {MIN_DURATION_S:.0f}s after trimming "
      f"(original duration was already short): {n_still_short}")
print(f"\nSaved trimmed copy to {CLEAN_DIR}")

trim pass: 100%|██████████| 155970/155970 [10:09<00:00, 255.97it/s]


Rows with zero VAD speech segments (left untrimmed, flagged): 31

Total audio: 188.99h -> 139.54h
Removed: 49.451h (1.14s/clip avg)
Clips still under 3s after trimming (original duration was already short): 21470

Saved trimmed copy to ../data/raw/openslr_52/processed/openslr_52_clean


### Audit the trim

Before/after for the exact clips flagged in the VAD audit above. The clips with the largest
original `vad_lead_sil_s` / `vad_trail_sil_s` get cut the most, so any buffer that's too tight
shows up here first. Also check the `vad_n_segments == 0` clips left untouched -- confirm
they're genuinely empty/noise rather than a short utterance VAD missed entirely.

In [18]:
for label, idxs in [("largest lead silence", worst_lead_idx), ("largest trail silence", worst_trail_idx)]:
    print(f"\n### {label}")
    for i in idxs:
        p = trim_audit_payload[i]
        print(f"\n[{i}] orig={p['orig_dur']:.2f}s -> trimmed={p['trim_dur']:.2f}s  "
              f"(lead={p['lead']:.2f}s / trail={p['trail']:.2f}s)  {meta_df.loc[i, 'text'][:80]!r}")
        print("  original:")
        display(Audio(filename=flac_path(i, DEDUP_DATA_DIR)))
        print("  trimmed:")
        display(Audio(filename=flac_path(i, CLEAN_DATA_DIR)))

print(f"\n### zero-speech clips (left untrimmed, {n_no_speech_trim} total)")
for i in no_speech_idx:
    p = trim_audit_payload[i]
    print(f"\n[{i}] {p['orig_dur']:.2f}s  text: {meta_df.loc[i, 'text'][:80]!r}")
    display(Audio(filename=flac_path(i, DEDUP_DATA_DIR)))


### largest lead silence

[1998607e4d] orig=26.70s -> trimmed=3.00s  (lead=24.60s / trail=0.70s)  'එය තීරණය වෙන්නේ,'
  original:


  trimmed:



[efbd5eb9ec] orig=26.80s -> trimmed=4.20s  (lead=21.30s / trail=1.90s)  'එවකට විර පරාක්\u200dරම නරේන්ද්\u200dරසිංහ රජතුමා'
  original:


  trimmed:



[cdc293228d] orig=25.80s -> trimmed=4.10s  (lead=20.90s / trail=1.40s)  'ඉන්දියානු රුපියල් කෝටි ක වියදමක් දරලා'
  original:


...


[695b9e248e] orig=27.60s -> trimmed=3.00s  (lead=2.00s / trail=23.40s)  'ගමේ මිනිසුන්ට මේ ආරංචිය ගියාම'
  original:


  trimmed:



### zero-speech clips (left untrimmed, 31 total)

[0d735b19fb] 3.50s  text: 'cool tamil'



[10087acc1e] 3.10s  text: 'මේ ගැන ලිපියක් පළ කෙරුවා'



[1184f95bdd] 4.40s  text: 'පාලකයින් බලපෑම් කරන්නේනම් එය වැරදියි'


### Remove zero-speech clips

`vad_n_segments == 0` clips were left untrimmed above and copied into `openslr_52_clean/`
unmodified -- VAD found no speech anywhere in them, so trimming on its own say-so would have
deleted the whole clip. Having audited a sample of them above, this removes them for good:
deletes the `.flac` from `openslr_52_clean/data/`, and drops the row from both
`utt_spk_text.tsv` and `meta.tsv`. `openslr_52_dedup/` (and the original raw source) are left
alone -- only the final clean copy loses these rows.

In [19]:
no_speech_ids = meta2_df.index[meta2_df["vad_n_segments"] == 0].tolist()

for file_id in no_speech_ids:
    p = flac_path(file_id, CLEAN_DATA_DIR)
    if os.path.exists(p):
        os.remove(p)

meta2_df = meta2_df.drop(index=no_speech_ids)
meta2_df.to_csv(CLEAN_META_PATH, sep="\t")

pd.DataFrame(
    [(fid, speaker_by_id[fid], meta_df.loc[fid, "text"]) for fid in meta2_df.index],
    columns=["file_id", "speaker_id", "transcript"],
).to_csv(CLEAN_TSV, sep="\t", header=False, index=False)

print(f"Removed {len(no_speech_ids)} zero-speech clips from {CLEAN_DIR}")
print(f"Remaining clips: {len(meta2_df):,}")

Removed 31 zero-speech clips from ../data/raw/openslr_52/processed/openslr_52_clean
Remaining clips: 155,939


### Verify the cleaned dataset

Confirms the file count on disk and TSV row count agree with how many rows were processed.

In [21]:
clean_flac_count = sum(len(files) for _, _, files in os.walk(CLEAN_DATA_DIR))
clean_tsv_rows = len(pd.read_csv(CLEAN_TSV, sep="\t", header=None))
print(f"{CLEAN_DIR}: {clean_flac_count:,} flac files, {clean_tsv_rows:,} tsv rows")
assert clean_flac_count == len(meta2_df) == clean_tsv_rows

../data/raw/openslr_52/processed/openslr_52_clean: 155,939 flac files, 155,939 tsv rows


## Final duration stats

Post-trim distribution, and how many clips still fall outside Whisper's 3-30s comfort band.

In [22]:
meta2_df["duration"].describe()

count    155939.000000
mean          3.220610
std           0.680122
min           1.100000
25%           3.000000
50%           3.000000
75%           3.300000
max          21.200000
Name: duration, dtype: float64

In [23]:
n_too_short = (meta2_df["duration"] < 3).sum()
n_too_long = (meta2_df["duration"] > 30).sum()

print(f"Clips < 3s: {n_too_short} ({n_too_short / len(meta2_df) * 100:.2f}%)")
print(f"Clips > 30s: {n_too_long} ({n_too_long / len(meta2_df) * 100:.2f}%)")
print(f"Total clips: {len(meta2_df)}")

Clips < 3s: 21458 (13.76%)
Clips > 30s: 0 (0.00%)
Total clips: 155939


## Pure English transcripts (informational)

Flags rows whose transcript contains no Sinhala script at all (Unicode block `U+0D80`-`U+0DFF`)
and at least one English letter -- e.g. `'cool tamil'` above, which is written in Latin script
despite the name. Informational only, nothing is dropped here.

In [26]:
import re

SINHALA_RE = re.compile(r"[඀-෿]")
ENGLISH_LETTER_RE = re.compile(r"[A-Za-z]")

texts = meta_df.loc[meta2_df.index, "text"].fillna("")
is_pure_english = ~texts.str.contains(SINHALA_RE) & texts.str.contains(ENGLISH_LETTER_RE)

n_pure_english = int(is_pure_english.sum())
print(f"Pure English rows: {n_pure_english:,} ({n_pure_english / len(meta2_df) * 100:.2f}%)")
texts[is_pure_english].head(10)

Pure English rows: 5,748 (3.69%)


file_id
00018c30ff                                day offices
000ab89be5                       magazines of the 90s
000bb8ae7d                                    android
00150b06a3              windows xp service pack 3 sp3
0021fa324d    full hindi movies with sinhala subtitle
0024b4aac4                        election department
0027c33491               world trade center sri lanka
002e2b91a8                                  nokla 525
003a027c8b                                  ethiopian
003d81317a                        harappa civiliation
Name: text, dtype: str

### Remove pure-English clips

This is a Sinhala ASR corpus -- rows flagged above as pure English don't belong in it. Deletes
the `.flac` from `openslr_52_clean/data/` and drops the row from both `utt_spk_text.tsv` and
`meta.tsv`, same as the zero-speech removal earlier. `openslr_52_dedup/` and the raw source are
left alone -- only the final clean copy loses these rows.

In [27]:
pure_english_ids = texts[is_pure_english].index.tolist()

for file_id in pure_english_ids:
    p = flac_path(file_id, CLEAN_DATA_DIR)
    if os.path.exists(p):
        os.remove(p)

meta2_df = meta2_df.drop(index=pure_english_ids)
meta2_df.to_csv(CLEAN_META_PATH, sep="\t")

pd.DataFrame(
    [(fid, speaker_by_id[fid], meta_df.loc[fid, "text"]) for fid in meta2_df.index],
    columns=["file_id", "speaker_id", "transcript"],
).to_csv(CLEAN_TSV, sep="\t", header=False, index=False)

print(f"Removed {len(pure_english_ids)} pure-English clips from {CLEAN_DIR}")
print(f"Remaining clips: {len(meta2_df):,}")

Removed 5748 pure-English clips from ../data/raw/openslr_52/processed/openslr_52_clean
Remaining clips: 150,191


## Denoise audition (optional)

Tries `noisereduce` on a sample of clips as an audition step only -- it's never applied to the
saved output, just listened to.

In [28]:
!pip install noisereduce

In [30]:
import noisereduce as nr

PROP_DECREASE = 0.8  # try 0.5 / 0.75 / 1.0 and compare by ear
N_NOISE_SAMPLE = 5


def denoise(path, prop_decrease=PROP_DECREASE):
    samples, sr = sf.read(path, dtype="float32", always_2d=True)
    mono = samples.mean(axis=1)
    reduced = nr.reduce_noise(y=mono, sr=sr, stationary=False, prop_decrease=prop_decrease)
    return reduced, sr


sample_ids = meta2_df.sample(min(N_NOISE_SAMPLE, len(meta2_df)), random_state=42).index

for file_id in sample_ids:
    path = flac_path(file_id, CLEAN_DATA_DIR)
    print(f"\nduration={meta2_df.loc[file_id, 'duration']:.2f}s  text: {meta_df.loc[file_id, 'text'][:80]!r}")
    print("  before:")
    display(Audio(filename=path))
    reduced, sr = denoise(path)
    print(f"  after (prop_decrease={PROP_DECREASE}):")
    display(Audio(data=reduced, rate=sr))


duration=4.50s  text: 'ඔයාට මේ වැඩේ වැඩි කාලයක් කරගෙන යන්න බැරිවෙයි.'
  before:


  after (prop_decrease=0.8):



duration=3.00s  text: 'රට බෙදීමේ යෝජනාවයි.'
  before:


  after (prop_decrease=0.8):



duration=3.00s  text: 'ඊට ඉස්සෙල්ලා අපි හිටියෙ දොළුකන්ද රක්ෂිතයේ'
  before:


  after (prop_decrease=0.8):



duration=3.00s  text: 'ඔහුගෙම සහොදරයන් විම'
  before:


  after (prop_decrease=0.8):



duration=3.00s  text: 'මොකද ඇය විවාහක නිසා.'
  before:


  after (prop_decrease=0.8):


In [2]:
import os
import random

import pandas as pd
from IPython.display import Audio, display

N_CHECK = 8  # how many random clips to audition
SEED = None  # set an int for a reproducible sample, or leave None for a fresh sample each run

CLEAN_DIR = "../../../../data/raw/openslr_52/processed/openslr_52_clean"
CLEAN_DATA_DIR = os.path.join(CLEAN_DIR, "data")
CLEAN_TSV = os.path.join(CLEAN_DIR, "utt_spk_text.tsv")


def flac_path(file_id, root):
    return os.path.join(root, file_id[:2], f"{file_id}.flac")


tsv_df = pd.read_csv(CLEAN_TSV, sep="\t", header=None, names=["file_id", "speaker_id", "text"], dtype=str)

rng = random.Random(SEED)
sample_ids = rng.sample(list(tsv_df["file_id"]), N_CHECK)

for file_id in sample_ids:
    row = tsv_df.loc[tsv_df["file_id"] == file_id].iloc[0]
    path = flac_path(file_id, CLEAN_DATA_DIR)

    print(f"\n[{file_id}]  speaker={row['speaker_id']}")
    print(f"  transcript: {row['text']!r}")

    if not os.path.exists(path):
        print(f"  MISSING FILE: {path}")
        continue

    display(Audio(filename=path))



[3660f478fe]  speaker=237ce
  transcript: 'මම ජනතාවගෙන් ඉල්ලා සිටිනවා.'



[5c882ff3d0]  speaker=06105
  transcript: 'එතකොට කොන්දොස්තරලා වෙන්න'



[36125a5dbc]  speaker=0766f
  transcript: 'ලෝක සත්ත්වයා හට තමාගේ යයි කියා කිසිවක් නැහැ.'



[4f0a4580a3]  speaker=dc5ef
  transcript: 'මේවා නරඹපු ප්\u200dරේක්ෂකයෝ'



[bf760e0f7b]  speaker=30026
  transcript: 'එයින් පළමුවැන්න හා අටවැන්න පමණක්'



[e884f8a5f2]  speaker=02b89
  transcript: 'මාගේ පැමිණීම ඔවුන්ට ප්\u200dරහේලිකාවක්'



[7e93bdaa0e]  speaker=3386d
  transcript: 'එහෙම බලලා හරියන්නෙත් නැහැ.'



[679f388259]  speaker=361fc
  transcript: 'තමයි කටයුතු කරන්නේ'


## Next steps

- The raw source (`data/raw/openslr_52/asr_sinhala/`) is untouched. Two new copies were written
  under `data/processed/`:
  - `openslr_52_dedup/` -- deduplicated audio (intermediate)
  - `openslr_52_clean/` -- deduplicated + VAD-trimmed audio + `utt_spk_text.tsv` + `meta.tsv`
    (per-file VAD stats and trim amounts) -- **this is the deliverable**.
- This replaces the parquet-based pipeline (`scripts/convert_openslr.py` -> `openslrData` ->
  `openslr_dedup.parquet` / `openslr_vad.parquet` / `openslr_trimmed.parquet`) for this dataset.
  Those parquet artifacts are no longer needed unless something downstream specifically wants
  the embedded-bytes format.
- To fold this into `combine_and_clean.ipynb`'s multi-source merge, either point its load step
  at `openslr_52_clean/` directly (reading loose `.flac` + TSV instead of parquet), or run
  `scripts/convert_openslr.py --openslr-dir ../data/processed/openslr_52_clean` once to produce
  a parquet shard from the cleaned tree -- `lingaData` (a partial 11,357-row conversion of this
  same corpus) is superseded either way.
- Denoising was only auditioned above, not applied -- decide from what you hear before wiring it
  into a saved pipeline step.